In [30]:
import cv2
from os import listdir
from os.path import isfile, join
from keras import backend as K
from keras.applications import ResNet50
from keras.models import Model
import numpy as np

from keras.preprocessing import image 


#from pretrained model: user anshu1106 on github

In [31]:
# Build model and load pretrained weights
model = ResNet50()

model.summary() # prints the summary of model with layer names and sizes

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,636,712 (97.80 MB)

 Trainable params: 25,583,592 (97.59 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [32]:
from keras.applications.resnet50 import preprocess_input, decode_predictions

# choose from one of the layer names 
layer_name = 'avg_pool'
emb_model = Model(inputs=model.input, outputs=model.get_layer(layer_name).output)

# Create a model which returns embedding at a particular layer 

In [33]:
from keras.preprocessing import image
def display_image(img_path):
    img = cv2.imread(img_path)
    cv_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(cv_rgb)
    plt.show()
    
def preprocess_input_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    return x

def predict_label(img_path):
    x = preprocess_input_image(img_path)
    prediction = model.predict(x)
    print(np.argmax(prediction))
    print('Predicted:', decode_predictions(prediction, top=3)[0])
    


In [34]:
img_path= '/Users/aditibelle/Desktop/pubmed_set/images/0a6fc26c-9cd9-4f95-8273-0a416457c92e.jpg'
test_image = preprocess_input_image(img_path)
# extract embedding
# embeddings=[]
embedding = emb_model.predict(test_image)
print(embedding)
# embeddings.append(embedding)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 636ms/step
[[0.07828413 0.06765559 0.         ... 0.13479476 0.06840472 0.06524852]]


In [35]:
img_path= '/Users/aditibelle/Desktop/pubmed_set/images/0a6fc26c-9cd9-4f95-8273-0a416457c92e.jpg'
from pathlib import Path
test = Path(img_path)
test.is_file()

True

In [36]:
import matplotlib.pyplot as plt                        
%matplotlib inline 
print(embedding.shape)

(1, 2048)


In [37]:
%%timeit
img_width = 224
img_height = 224
#vgg_path = '/Users/gene/Learn/keras-rtst/vgg16_weights.h5'
images_path = '/Users/aditibelle/Desktop/pubmed_set/images/'
# tsne_path = '/Users/anshu/meet-up/internship/embedding/tsne_points.txt'

# get images
images = [f for f in listdir(images_path) if isfile(join(images_path, f))]
images_all = [images[i]  for i in range(1,len(images), 1) ] 

images =[]
for im in images_all:
    if(im!='.DS_Store'):
        images.append(im) 
#print(images)        

features = np.zeros([len(images), 2048])
i=0
for im in images[:2]:
    if(im=='.DS_Store'):
        continue
    test_image = preprocess_input_image(images_path+im)
    
    embedding = emb_model.predict(test_image)
    features[i, :] = emb_model.predict(test_image)
    i=i+1



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━

In [39]:
no_of_images=(len(images), 1)
print(no_of_images)
import pickle
with open("features", 'wb') as f:
    pickle.dump((features, images), f)

NameError: name 'images' is not defined

In [40]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=0)
reduced = tsne.fit_transform(features)

NameError: name 'features' is not defined

In [ ]:
# plt.figure(figsize=(8, 8))
# plt.scatter(x = reduced[:,0], y=reduced[:,1])
# plt.show()
from collections import Counter

vis_x = reduced[:, 0]
vis_y = reduced[:, 1]

plt.scatter(vis_x, vis_y, cmap=plt.cm.get_cmap("jet", 10))
plt.colorbar(ticks=range(10))
plt.clim(-0.5, 9.5)
plt.show()

In [ ]:
reduced_transformed = reduced - np.min(reduced, axis=0)
reduced_transformed /= np.max(reduced_transformed, axis=0)
image_xindex_sorted = np.argsort(np.sum(reduced_transformed, axis=1))
print(image_xindex_sorted.shape)

In [24]:
# draw all images in a merged image
no_of_images=50
from PIL import Image
image_width = 500
merged_width = int(np.ceil(np.sqrt(no_of_images))*image_width)
merged_image = np.zeros((merged_width, merged_width, 3), dtype='uint8')
ellipside =True
for counter, index in enumerate(image_xindex_sorted):
    # set location
    if ellipside:
        a = np.ceil(reduced_transformed[counter, 0] * (merged_width-image_width-1)+1)
        b = np.ceil(reduced_transformed[counter, 1] * (merged_width-image_width-1)+1)
        a = int(a - np.mod(a-1,image_width) + 1)
        b = int(b - np.mod(b-1,image_width) + 1)
        if merged_image[a,b,0] != 0:
            continue
        image_address = images[counter]
        img = np.asarray(Image.open('Desktop/pubmed_set/images/'+image_address).resize((image_width, image_width)))
        merged_image[a:a+image_width, b:b+image_width,:] = img[:,:,:3]
    else:
        b = int(np.mod(counter, np.sqrt(no_of_images)))
        a = int(np.mod(counter//np.sqrt(no_of_images), np.sqrt(no_of_images)))
        image_address = images[index]
        print(image_address)
        img = np.asarray(Image.open('Desktop/pubmed_set/images/'+image_address).resize((image_width, image_width)))
        merged_image[a*image_width:(a+1)*image_width, b*image_width:(b+1)*image_width,:] = img[:,:,:3]

plt.imshow(merged_image)
plt.show()
merged_image = Image.fromarray(merged_image)
if ellipside:
    merged_image.save('merged-%s-ellipsoide-resnet50.png'%images_path.split('/')[-2])
else:
    merged_image.save('merged-%s.png'%images_path.split('/')[-2])

NameError: name 'image_xindex_sorted' is not defined

In [25]:
img_path= '/Users/aditibelle/Desktop/pubmed_set/images/0a6fc26c-9cd9-4f95-8273-0a416457c92e.jpg'
from pathlib import Path
test = Path('/Users/aditibelle/Dimages/bfce72df-d774-4641-b1f9-7849efa080ad.jpg')
test.is_file()

False

In [26]:
#from tensorflow.contrib.tensorboard.plugins import projector

import tensorflow as tf
from tensorboard.plugins import projector

In [27]:
embeddings_path = '/Users/aditibelle/Desktop/Embeddings'
LOG_DIR = '/Users/aditibelle/Desktop/Embeddings_Log'

In [28]:
features.shape


NameError: name 'features' is not defined

In [29]:
np.savetxt('feature_vectors_400_samples.txt',features)
#feature_vectors = np.loadtxt('feature_vectors.txt')
pickle.dump(features, open('feature_vectors_400_samples.pkl', 'wb'))

NameError: name 'features' is not defined

In [ ]:
img_data=[]
print(images)



In [ ]:
for img in images:
    input_img=cv2.imread('Desktop/pubmed_set/images/'+img )
    input_img_resize=cv2.resize(input_img,(224,224))
    img_data.append(input_img_resize)
    
                
img_data = np.array(img_data)


In [ ]:
import tensorflow as tf
import os
feature_vectors = np.loadtxt('feature_vectors_400_samples.txt')
print ("feature_vectors_shape:",feature_vectors.shape)
print ("num of images:",feature_vectors.shape[0])
print ("size of individual feature vector:",feature_vectors.shape[1])
features = tf.Variable(feature_vectors, name='features')
##images_name = ['parrot','dog','cat','gun','fruit','car','face','box','others'] adjust for tissue types


metadata_file = open(os.path.join(LOG_DIR, 'metadata_4_classes.tsv'), 'w')
metadata_file.write('Class\tName\n')

for i,label in enumerate(target):
    c = images_name[target[i]-1]
    metadata_file.write('{}\t{}\n'.format(target[i],c))
    
metadata_file.close()

In [ ]:
def images_to_sprite(data):
    """Creates the sprite image along with any necessary padding

    Args:
      data: NxHxW[x3] tensor containing the images.

    Returns:
      data: Properly shaped HxWx3 image with any necessary padding.
    """
    if len(data.shape) == 3:
        data = np.tile(data[...,np.newaxis], (1,1,1,3))
    data = data.astype(np.float32)
    min = np.min(data.reshape((data.shape[0], -1)), axis=1)
    data = (data.transpose(1,2,3,0) - min).transpose(3,0,1,2)
    max = np.max(data.reshape((data.shape[0], -1)), axis=1)
    data = (data.transpose(1,2,3,0) / max).transpose(3,0,1,2)
    # Inverting the colors seems to look better for MNIST
    #data = 1 - data

    n = int(np.ceil(np.sqrt(data.shape[0])))
    padding = ((0, n ** 2 - data.shape[0]), (0, 0),
            (0, 0)) + ((0, 0),) * (data.ndim - 3)
    data = np.pad(data, padding, mode='constant',
            constant_values=0)
    # Tile the individual thumbnails into an image.
    data = data.reshape((n, n) + data.shape[1:]).transpose((0, 2, 1, 3)
            + tuple(range(4, data.ndim + 1)))
    data = data.reshape((n * data.shape[1], n * data.shape[3]) + data.shape[4:])
    data = (data * 255).astype(np.uint8)
    return data
#%%
sprite = images_to_sprite(img_data)
cv2.imwrite(os.path.join(LOG_DIR, 'sprite_4_classes.png'), sprite)

In [ ]:
with tf.Session() as sess:
    saver = tf.train.Saver([features])

    sess.run(features.initializer)
    saver.save(sess, os.path.join(LOG_DIR, 'images_4_classes.ckpt'))
    
    config = projector.ProjectorConfig()
    # One can add multiple embeddings.
    embedding = config.embeddings.add()
    embedding.tensor_name = features.name
    # Link this tensor to its metadata file (e.g. labels).
    embedding.metadata_path = os.path.join(LOG_DIR, 'metadata_4_classes.tsv')
    # Comment out if you don't want sprites
    embedding.sprite.image_path = os.path.join(LOG_DIR, 'sprite_4_classes.png')
    embedding.sprite.single_image_dim.extend([img_data.shape[1], img_data.shape[1]])
    # Saves a config file that TensorBoard will read during startup.
    projector.visualize_embeddings(tf.summary.FileWriter(LOG_DIR), config)